<a href="https://colab.research.google.com/github/Octaxx/DLI-Assignment/blob/main/GUI.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
!pip -q install --upgrade pip
!pip -q install gradio joblib scikit-learn pyspellchecker tensorflow

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.8/1.8 MB 45.2 MB/s eta 0:00:00


In [2]:
from google.colab import drive
drive.mount('/content/drive')
MODEL_PATH = "/content/drive/MyDrive/Colab Notebooks/PhishingModel/Gavin/GavinModel.pkl"
print("Model path:", MODEL_PATH)


Mounted at /content/drive
Model path: /content/drive/MyDrive/Colab Notebooks/PhishingModel/Gavin/GavinModel.pkl


In [3]:
import re, sys, types, numpy as np
from sklearn.base import BaseEstimator, TransformerMixin

# --- sklearn compatibility shim (for pickles referring to 'sklearn.ensemble_voting')
try:
    import sklearn.ensemble as _sk_ens  # noqa: F401
    if "sklearn.ensemble_voting" not in sys.modules:
        shim = types.ModuleType("sklearn.ensemble_voting")
        from sklearn.ensemble import VotingClassifier, VotingRegressor
        shim.VotingClassifier = VotingClassifier
        shim.VotingRegressor = VotingRegressor
        sys.modules["sklearn.ensemble_voting"] = shim
except Exception:
    pass

# Optional: pyspellchecker (safe fallback to 0 if not present)
try:
    from spellchecker import SpellChecker
    _spell = SpellChecker()
except Exception:
    _spell = None

_STOPWORDS = set(['the','is','in','and','to','of','for','on','with','that','this','it','as','at','by','an','be'])

def _char_count(t): return len(t)
def _word_count(t): return len(re.findall(r'\b\w+\b', t))
def _exclam_count(t): return t.count('!')
def _uppercase_ratio(t):
    upp = sum(1 for c in t if c.isupper()); return (upp / max(1, len(t)))
def _has_link(t): return int(bool(re.search(r'http[s]?://', t, flags=re.I)))
def _has_login_word(t): return int('login' in t.lower())
def _has_html(t): return int(bool(re.search(r'<[^>]+>', t)))
def _url_count(t): return len(re.findall(r'http[s]?://', t, flags=re.I))
def _num_dots(t): return t.count('.')
def _has_attachment_word(t): return int(bool(re.search(r'attachment|invoice|pdf', t, flags=re.I)))
def _has_suspicious_word(t): return int(bool(re.search(r'verify|account|confirm|login', t, flags=re.I)))
def _avg_word_len(t):
    wc = _word_count(t); return (len(t) / (wc + 1))
def _special_char_count(t): return len(re.findall(r'[#$%^&*()]', t))
def _has_spam_phrase(t): return int(bool(re.search(r'congratulations|you have won|limited time|click below', t, flags=re.I)))
def _has_ip_in_url(t): return int(bool(re.search(r'http[s]?://(?:\d{1,3}\.){3}\d{1,3}', t)))
def _domain_length(t):
    urls = re.findall(r'http[s]?://([^\s/]+)', t)
    return (np.mean([len(u) for u in urls]) if urls else 0)
def _spelling_errors(t):
    if _spell is None: return 0
    words = re.findall(r'\b[a-zA-Z]{2,}\b', t)
    return len(_spell.unknown(words))
def _stopword_ratio(t):
    words = re.findall(r'\b[a-zA-Z]{2,}\b', t.lower())
    return (sum(1 for w in words if w in _STOPWORDS) / (len(words) or 1))
def _text_entropy(t):
    if not t: return 0.0
    counts = {}
    for c in t: counts[c] = counts.get(c,0)+1
    probs = [cnt/len(t) for cnt in counts.values()]
    return float(-sum(p*np.log2(p) for p in probs if p > 0))
def _iframe_count(t): return len(re.findall(r'<iframe', t, flags=re.I))
def _form_count(t):   return len(re.findall(r'<form', t, flags=re.I))
def _html_to_text_ratio(t):
    tags = re.findall(r'<.*?>', t)
    return (len("".join(tags)) / (len(t)+1))
def _cue_features(t):
    L = t.lower()
    return {
        'has_urgent':  int(any(w in L for w in ['urgent','immediate','now'])),
        'has_threat':  int(any(w in L for w in ['suspend','locked','compromised'])),
        'has_generic': int(any(w in L for w in ['dear user','dear customer','valued customer'])),
    }

# ---- HybridFeatureBuilder using tf.keras (not standalone keras)
class HybridFeatureBuilder(BaseEstimator, TransformerMixin):
    """
    Rebuilds hybrid features from raw email text using:
      - tokenizer (Keras)
      - LSTM (JSON + weights) as a feature extractor (pre-final layer)
      - scalers for statistical and cue features
    """
    def __init__(self, tokenizer, max_len, lstm_json, lstm_weights, stat_scaler, cue_scaler):
        self.tokenizer = tokenizer
        self.max_len = max_len
        self.lstm_json = lstm_json
        self.lstm_weights = lstm_weights
        self.stat_scaler = stat_scaler
        self.cue_scaler = cue_scaler
        self._model = None  # lazy restore

    def _restore_lstm(self):
        from tensorflow.keras.models import model_from_json
        from tensorflow.keras import Model
        if self._model is None:
            base = model_from_json(self.lstm_json)
            base.set_weights(self.lstm_weights)
            lstm_layer = None
            for lyr in base.layers[::-1]:
                if "lstm" in lyr.name.lower():
                    lstm_layer = lyr; break
            if lstm_layer is None:
                raise RuntimeError("No LSTM layer found during restore.")
            self._model = Model(inputs=base.input, outputs=lstm_layer.output)
        return self._model

    def fit(self, X, y=None):
        return self

    def transform(self, X):
        from tensorflow.keras.preprocessing.sequence import pad_sequences
        texts = [str(t) if t is not None else "" for t in X]

        # stat & cue features
        stat_vecs, cue_vecs = [], []
        for t in texts:
            stat = np.array([
                _char_count(t), _word_count(t), _exclam_count(t), _uppercase_ratio(t),
                _has_link(t), _has_login_word(t), _has_html(t), _url_count(t), _num_dots(t),
                _has_attachment_word(t), _has_suspicious_word(t), _avg_word_len(t),
                _special_char_count(t), _has_spam_phrase(t),
                _has_ip_in_url(t), _domain_length(t), _spelling_errors(t), _stopword_ratio(t),
                _text_entropy(t), _iframe_count(t), _form_count(t), _html_to_text_ratio(t),
            ], dtype=float)
            c = _cue_features(t)
            cue = np.array([c['has_urgent'], c['has_threat'], c['has_generic']], dtype=float)
            stat_vecs.append(stat); cue_vecs.append(cue)

        stat_vecs = np.vstack(stat_vecs)
        cue_vecs  = np.vstack(cue_vecs)
        stat_s = self.stat_scaler.transform(stat_vecs)
        cue_s  = self.cue_scaler.transform(cue_vecs)

        # LSTM vector
        seqs = self.tokenizer.texts_to_sequences(texts)
        pads = pad_sequences(seqs, maxlen=self.max_len)
        lstm_v = self._restore_lstm().predict(pads, verbose=0)

        # concat -> [lstm | stat_s | cue_s]
        return np.hstack([lstm_v, stat_s, cue_s])

    def __getstate__(self):
        state = self.__dict__.copy()
        state['_model'] = None
        return state

    def __setstate__(self, state):
        self.__dict__.update(state)
        self._model = None


In [7]:
import joblib, gradio as gr

# Load your pipeline (now that HybridFeatureBuilder exists in scope)
model = joblib.load(MODEL_PATH)

def predict_email(text: str):
    text = (text or "").strip()
    if not text:
        return "⚠ Enter email text", ""
    try:
        pred = int(model.predict([text])[0])  # 1 = phishing, 0 = legit
        label = "PHISHING" if pred == 1 else "LEGITIMATE"
        try:
            proba = float(model.predict_proba([text])[0][1])
            score = f"{proba*100:.2f}%"
        except Exception:
            score = "N/A"
        return label, score
    except Exception as e:
        return f"Prediction error: {e}", ""

demo = gr.Interface(
    fn=predict_email,
    inputs=gr.Textbox(lines=12, placeholder="Paste email text here..."),
    outputs=[gr.Label(label="Prediction"), gr.Textbox(label="P(Phishing)")],
    title="Phishing Email Text Detector",
    description="Paste any email text. The model predicts PHISHING vs LEGITIMATE and probability."
)

demo.launch(share=True)


Colab notebook detected. To show errors in colab notebook, set debug=True in launch()
* Running on public URL: https://2aaed17181796fe0fe.gradio.live

This share link expires in 1 week. For free permanent hosting and GPU upgrades, run `gradio deploy` from the terminal in the working directory to deploy to Hugging Face Spaces (https://huggingface.co/spaces)
